## Section 1 — Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

from src.utils import prepare_training_frame

# Load data
DATA_PATH = "data/data.csv"
df = pd.read_csv(DATA_PATH)

# Build features/target using shared project logic
X, y = prepare_training_frame(df)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

y_train.value_counts(normalize=True).rename({0: "No", 1: "Yes"})

## Section 2 — Baseline (DummyClassifier)

In [ ]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy="most_frequent", random_state=42)
baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)
baseline_proba = baseline.predict_proba(X_test)[:, 1]

baseline_metrics = {
    "Model": "DummyClassifier(most_frequent)",
    "Accuracy": accuracy_score(y_test, baseline_pred),
    "ROC-AUC": roc_auc_score(y_test, baseline_proba),
}

baseline_metrics

## Section 3 — Model comparison

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from src.pipeline.train import build_training_pipeline

candidates = [
    (
        "LogisticRegression",
        LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    ),
    (
        "RandomForestClassifier",
        RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42),
    ),
    (
        "GradientBoostingClassifier",
        GradientBoostingClassifier(n_estimators=100, random_state=42),
    ),
]

rows = []
trained_pipelines = {}

for name, estimator in candidates:
    pipe = build_training_pipeline(estimator=estimator)
    pipe.fit(X_train, y_train)

    proba = pipe.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)

    rows.append(
        {
            "Model": name,
            "Accuracy": accuracy_score(y_test, pred),
            "ROC-AUC": roc_auc_score(y_test, proba),
        }
    )
    trained_pipelines[name] = pipe

comparison_df = pd.DataFrame(rows).sort_values("ROC-AUC", ascending=False).reset_index(drop=True)
comparison_df

## Section 4 — Best model evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_curve

best_model_name = comparison_df.loc[0, "Model"]
best_pipe = trained_pipelines[best_model_name]

best_proba = best_pipe.predict_proba(X_test)[:, 1]
best_pred = (best_proba >= 0.5).astype(int)

print("Best model by ROC-AUC:", best_model_name)
print("Accuracy:", accuracy_score(y_test, best_pred))
print("ROC-AUC:", roc_auc_score(y_test, best_proba))

# Confusion matrix
cm = confusion_matrix(y_test, best_pred)
plt.figure(figsize=(4.5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title(f"Confusion matrix — {best_model_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

# Classification report
print("\nClassification report")
print(classification_report(y_test, best_pred, target_names=["No", "Yes"]))

# ROC curve
fpr, tpr, _ = roc_curve(y_test, best_proba)
auc = roc_auc_score(y_test, best_proba)
plt.figure(figsize=(5, 4))
plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.title(f"ROC curve — {best_model_name}")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

# Feature importance / coefficients
pre = best_pipe.named_steps["preprocessor"]
clf = best_pipe.named_steps["classifier"]

# Get feature names after preprocessing
feature_names = []

# numeric transformer
try:
    feature_names.extend(pre.get_feature_names_out())
except Exception:
    # Fallback for older sklearn / mixed transformers
    num_features = pre.transformers_[0][2]
    cat_features = pre.transformers_[1][2]
    ohe = pre.named_transformers_["categorical"].named_steps["encoder"]
    cat_names = list(ohe.get_feature_names_out(cat_features))
    feature_names = list(num_features) + cat_names

if hasattr(clf, "feature_importances_"):
    importances = pd.Series(clf.feature_importances_, index=feature_names).sort_values(ascending=False)
    top15 = importances.head(15)[::-1]
    plt.figure(figsize=(8, 5))
    sns.barplot(x=top15.values, y=top15.index, palette="viridis")
    plt.title(f"Top 15 feature importances — {best_model_name}")
    plt.xlabel("Importance")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()
elif hasattr(clf, "coef_"):
    coefs = pd.Series(clf.coef_.ravel(), index=feature_names).sort_values()
    top_pos = coefs.tail(15)
    top_neg = coefs.head(15)
    top = pd.concat([top_neg, top_pos]).sort_values()

    plt.figure(figsize=(8, 6))
    sns.barplot(x=top.values, y=top.index, palette="coolwarm")
    plt.axvline(0, color="black", linewidth=1)
    plt.title(f"Top coefficients (±) — {best_model_name}")
    plt.xlabel("Coefficient")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()
else:
    print("Model does not expose feature importances or coefficients.")

## Section 5 — Conclusion

The best model in this comparison is chosen by **ROC-AUC** because churn prediction is a **class-imbalanced** problem and ROC-AUC evaluates ranking quality across thresholds. The preprocessing pipeline uses median imputation + scaling for numeric features and most-frequent imputation + one-hot encoding for categorical features. For class imbalance handling, the logistic regression and random forest runs use `class_weight="balanced"`, while gradient boosting is evaluated without class weighting.